# Lesson 5.1 — Closing the Loop: Tracking Error and Success Criteria

Execute produces motion; **Track** judges it against explicit criteria (final error, RMS, pose) and returns a verdict with a localising reason.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand, to_configuration, plan_reference, execute_reference, track,
    system_monitor, fk_xy, P2, T2)
def pick_layer(seed_world=1, seed_perc=7):
    w = GreenhouseWorld.demo_row(n=6, seed=seed_world)
    dets = model_perception(w, rng=np.random.default_rng(seed_perc))
    _, tgt = understand(dets, w)
    layer = plan_reference(w.q.copy(), to_configuration(tgt), rng=np.random.default_rng(0))
    return w, tgt, layer
checks = []
w, tgt, layer = pick_layer()


### Healthy pick -> Track says success

In [2]:
rec = execute_reference(layer)
v = track(rec, target=tgt)
print('verdict:', v['success'], '| reason:', v['reason'],
      '| rms=%.4f pose_err=%.4f' % (v['rms'], v['pose_error']))
checks.append(v['success'] is True and v['reason'] == 'ok')


verdict: True | reason: ok | rms=0.0001 pose_err=0.0000


### Strong disturbance -> Track says failure AND names the criterion

In [3]:
def big(t, j): return 60.0 if (j == 0 and t > 0.3) else 0.0
rec_f = execute_reference(layer, disturbance=big)
vf = track(rec_f, target=tgt)
print('verdict:', vf['success'], '| reason:', vf['reason'],
      '| final_err_max=%.3f' % vf['final_error_max'])
checks.append(vf['success'] is False)
checks.append(vf['reason'] in ('final_error', 'rms', 'pose'))   # localises the failed criterion


verdict: False | reason: final_error | final_err_max=2.020


### reached vs succeeded: the verdict is owned by Track, not the controller

In [4]:
print('Execute reported reached =', rec['reached'], '(joints settled near reference end)')
print('Track  reported success =', v['success'], '(criteria met: error, rms, pose)')
checks.append(rec['reached'] and v['success'])   # both true on a healthy pick
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


Execute reported reached = True (joints settled near reference end)
Track  reported success = True (criteria met: error, rms, pose)
4/4 checks passed.
All checks passed.
